### Middlewares

Middlewares provide a way to more tightly control what happens inside the agent. Middleware is useful for the following:
-Tracking agent behaviour with logging, analytics, and debugging.
-Trasnforming prompt, tool selection, and output formatting.
-Adding retries, fallbacks, and early termination logic.
-Applying rate limits, guardrails, and PII detection.

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


### Summarization Middleware

In [18]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# Load environment variables
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Main model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# Create agent with memory + summarization
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

# Thread configuration
config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}

# Test questions
questions = [
    "What is 11-2?",
    "What is 100/5?",
    "What is 12+2?",
    "What is 3*7?",
    "What is 2/2?",
    "What is 4+2?",
]

# Invoke agent repeatedly
for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

    print(f"Question: {q}")
    print(f"Answer: {response['messages'][-1].content}")
    print(f"Total Messages: {len(response['messages'])}")
    print("-" * 50)

Question: What is 11-2?
Answer: 11 - 2 = 9
Total Messages: 2
--------------------------------------------------
Question: What is 100/5?
Answer: 100 / 5 = 20
Total Messages: 4
--------------------------------------------------
Question: What is 12+2?
Answer: 12 + 2 = 14
Total Messages: 6
--------------------------------------------------
Question: What is 3*7?
Answer: 3 * 7 = 21
Total Messages: 8
--------------------------------------------------
Question: What is 2/2?
Answer: 2 / 2 = 1
Total Messages: 10
--------------------------------------------------
Question: What is 4+2?
Answer: 4 + 2 = 6
Total Messages: 6
--------------------------------------------------
